# Text-based Machine Learning

---

> Machine Learning is “A field of computer science that gives computers the ability to learn from data without being explicitly programmed.” — Wikipedia

### Types of ML:
| Type                  | Description                                                | Examples                          |
|-----------------------|------------------------------------------------------------|-----------------------------------|
| **Supervised**        | Data has known labels; learn input-output mapping          | Classification, Regression        |
| **Unsupervised**      | No labels; discover structure from data                    | Clustering, Topic Modeling        |
| **Reinforcement**     | Learn by interacting with environment to maximize reward   | Game AI, Robotics                 |

---

# Feature Engineering Overview
> Feature enginnering: the process of converting text into vectors (numbers)

> Foundation: Vector Space Model (a.k.a Term Vector Model), vsm for short, is a mathmatical model for representing text documents as vectors in a multi-dimensional space.这是后续几个模型的理论基础，把文档表示为为不同维的向量，每一个维度表示一个词，**值**表示这个词在文档中的权重（例如：词频frequency, 出现次数count, TF-IDF, 是否出现binary(0/1)）
- Documents are represented as vectors:
  `D_j = [w1j, w2j, ..., wnj]`
- Each dimension is a **term (word)**; each value is the **weight** of that term in the document (e.g., frequency, TF-IDF)

---

### 1. Bag-of-Words (BoW) Model - unigram

- Ignores grammar, order, punctuation — treats document as a “bag” of words
- Creates a Document-Term Matrix (DTM)
- Can be built with:
  - **Keras**: `texts_to_matrix()` with modes like `binary`, `count`, `freq`, `tfidf`
  - **Scikit-learn**: `CountVectorizer()` ➝ sparse matrix

    ### 🔢 Keras Example
    ```python
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(sentences)
    word_bag = tokenizer.texts_to_matrix(sentences, mode='binary')

### 2. N-Gram Model
- Extension of BoW to sequences of n words
- Common n-grams: bigrams (2-grams), trigrams (3-grams)
- Implemented with CountVectorizer(ngram_range=(n, n))



Example:

If corpus is: `["I love cats", "I hate cats"]` \
vocabulary = `{'I': 1, 'love': 2, 'hate': 3, 'cats': 4}` \
Then the document-term matrix is (unigram / bi-gram / trigram / N-gram):
```
|      | I | love | hate | cats |
|------|---|------|------|------|
| 1    | 1 | 1    | 0    | 1    |
| 2    | 1 | 0    | 1    | 1    |
```
And `I love cats` becomes `[1, 1, 0, 1]` and `I hate cats` becomes `[1, 0, 1, 1]`

# Different modes
## TF-IDF (Term Frequency-Inverse Document Frequency)
- Measures importance of a term in a document relative to a corpus
- TF-IDF = TF * IDF
> TF = term frequency (how often a term appears in a document)
> IDF = inverse document frequency (how common or rare a term is across all documents)
> 即「词在当前文档中出现得多 × 在所有文档中不常见」

### Tfidf的三种实现方式
- From **keras** Tokenizer: It is by default unigram
    - ```python
      from keras.preprocessing.text import Tokenizer
      word_bag = tokenizer.texts_to_matrix(sentence_list, mode='tfidf')
      ```

- From **sklearn** CountVectorizer
    - ```python
      from sklearn.feature_extraction.text import CountVectorizer
      cv = CountVectorizer(min_df=0., max_df=1.) # min/max_df: min/max frequency of a token
      # 可以用min_df=0.1, max_df=0.9来过滤掉一些高频词和低频词
      ```
- From **sklearn** TfidfVectorizer
    - ```python
      from sklearn.feature_extraction.text import TfidfVectorizer
      vectorizer = TfidfVectorizer()
      unigram_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
      bigram_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
      # n-gram_range=(1, 2) means unigram + bigram
      # (1,1) means unigram only
      # (2,2) means bigram only
      # (1,3) means unigram + bigram + trigram
      ```


# Different representation - in BoW
### sparse matrix 稀疏矩阵
每一行表示一个 (行号, 列号) 值，也就是：\
所有非0（row, column) value）\
“第几篇文档的第几个词出现了几次”
### dense matrix 密集矩阵
每一行就是一个文档的词向量（Document-Term Vector）

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = ["I love NLP NLP NLP", "NLP is fun", "I love machine learning"]

# 初始化 CountVectorizer（使用 unigram）
vectorizer = CountVectorizer()

# 拟合并转换为稀疏矩阵
sparse_matrix = vectorizer.fit_transform(corpus)

# 查看词汇表（列名）
print("Vocabulary (Features):")
print(vectorizer.get_feature_names_out())

# 打印稀疏矩阵
print("\nSparse Matrix (compressed format):")
print(sparse_matrix)

"""
  (0, 3)  1
  (0, 5)  1
  (1, 0)  1
  (1, 1)  1
  (1, 5)  1
  (2, 2)  1
  (2, 3)  1
  (2, 4)  1
  (2, 5)  1
"""

# 转换成密集矩阵
dense_matrix = sparse_matrix.toarray()

"""
[
  [0, 0, 0, 1, 0, 1],  # "I love NLP"
  [1, 1, 0, 0, 0, 1],  # "NLP is fun"
  [0, 0, 1, 1, 1, 1]   # "I love machine learning"
]
"""
# 打印密集矩阵
print("\nDense Matrix (normal array):")
print(dense_matrix)


Vocabulary (Features):
['fun' 'is' 'learning' 'love' 'machine' 'nlp']

Sparse Matrix (compressed format):
  (0, 3)	1
  (0, 5)	3
  (1, 5)	1
  (1, 1)	1
  (1, 0)	1
  (2, 3)	1
  (2, 4)	1
  (2, 2)	1

Dense Matrix (normal array):
[[0 0 0 1 0 3]
 [1 1 0 0 0 1]
 [0 0 1 1 1 0]]




# Advanced Feature Engineering
## Word2Vec
### Why Word2Vec?
传统的 TF-IDF / Bag-of-Words 有两个主要缺点：
| 问题 | 原因 |
|------|------|
| ❌ 无法理解上下文 | 比如 “银行” 和 “河岸” 都是同一个词 |
| ❌ 特征稀疏、维度高 | 每个词一个维度，词表大就爆炸了 |

- ✅ We need:
> **将词变成有“语义”的、低维度的密集向量（word embeddings）** convert words into low-dimensional dense vectors with semantic meaning

- 🧠 Fundamental Theory：Distributional Hypothesis

> “You shall know a word by the company it keeps.” 词的意思可以由它周围的词来决定。

---

- Word2Vec is a technique to represent words as vectors in a continuous vector space
- It captures semantic meaning and relationships between words
- Trained on large corpus of text
- Two main architectures:
  - Continuous Bag of Words (CBOW): Predicts target word from context words
  - Skip-gram: Predicts context words from target word
    | 模型 | 作用 | 举例（中心词为 brown） |
    |------|------|----------------|
    | **CBOW** | 用上下文预测目标词 | 输入：["quick", "fox"] → 输出："brown" |
    | **Skip-Gram** | 用目标词预测上下文 | 输入："brown" → 输出：["quick", "fox"] |

- CBOW 是听别人说话猜中间词
- Skip-Gram 是说一个词，猜你左右的词

---

## ✨ CBOW 实现流程（Slides 12-22）

1. 使用 Keras 的 `Tokenizer` 生成词典和 `word2id`
2. 构造 `(context, target)` 对
3. 构建神经网络结构：
   - `Embedding` 层：将词 ID 转成向量
   - `Lambda` 层：对 context 词向量求平均
   - `Dense + softmax`：输出最有可能的目标词
4. 训练模型：遍历语料训练 `(X, y)` 样本
5. 提取词向量：`cbow.get_weights()[0]` 就是 embedding matrix

---

## ✨ Skip-Gram 实现流程（Slides 23-30）

1. 使用 `skipgrams()` 构造 `(target, context)` 对
2. 构建双输入神经网络：
   - 两个 `Embedding` 层分别处理 target 和 context
   - `Dot` 层计算两个词的相似度（内积）
   - `Dense + sigmoid` 输出是否是好搭配
3. 训练模型，提取 embedding 层的权重

---

## 🔍 相似词分析 & 可视化（Slides 21 & 30）

使用欧几里得距离来分析词之间的关系：

```python
from sklearn.metrics.pairwise import euclidean_distances

🌟想实际用的话，现在大家常用 gensim.models.Word2Vec 直接训练，或者直接加载 GoogleNews 的预训练模型。

In [ ]:
"""CROW implementation"""
# ✅ Step 1: 预处理与词表
from keras.preprocessing import text, sequence
from keras.utils import to_categorical

tokenizer = text.Tokenizer()
tokenizer.fit_on_texts(norm_tale)

word2id = tokenizer.word_index
id2word = {v: k for k, v in word2id.items()}
word2id['PAD'] = 0

wids = [[word2id[w] for w in text.text_to_word_sequence(doc)] for doc in norm_tale]
vocab_size = len(word2id)
embed_size = 100
window_size = 2

# ✅ Step 2: 生成 (context, target) 对
def generate_context_word_pairs(corpus, window_size, vocab_size):
    context_len = window_size * 2
    for words in corpus:
        for index, word in enumerate(words):
            start = index - window_size
            end = index + window_size + 1
            context = [words[i] for i in range(start, end) if 0 <= i < len(words) and i != index]
            target = word
            x = sequence.pad_sequences([context], maxlen=context_len)
            y = to_categorical([target], vocab_size)
            yield x, y

# ✅ Step 3: 建立模型
from keras.models import Sequential
from keras.layers import Dense, Embedding, Lambda
import keras.backend as K

cbow = Sequential()
cbow.add(Embedding(input_dim=vocab_size, output_dim=embed_size, input_length=window_size*2))
cbow.add(Lambda(lambda x: K.mean(x, axis=1), output_shape=(embed_size,)))
cbow.add(Dense(vocab_size, activation='softmax'))

cbow.compile(loss='categorical_crossentropy', optimizer='rmsprop')

# ✅ Step 4: 训练
for epoch in range(1, 2):
    loss = 0.
    for x, y in generate_context_word_pairs(wids, window_size, vocab_size):
        loss += cbow.train_on_batch(x, y)
    print('Epoch:', epoch, '\tLoss:', loss)

# ✅ Step 5: 获取词向量
weights = cbow.get_weights()[0][1:]  # skip PAD


In [ ]:
"""Skip-Gram implementation"""
# ✅ Step 1: 预处理与词表
from keras.preprocessing import text, sequence
from keras.preprocessing.sequence import skipgrams

tokenizer = text.Tokenizer()
tokenizer.fit_on_texts(norm_tale)

word2id = tokenizer.word_index
id2word = {v: k for k, v in word2id.items()}
vocab_size = len(word2id) + 1  # 预留 index 0

wids = [[word2id[w] for w in text.text_to_word_sequence(doc)] for doc in norm_tale]
embed_size = 100

# ✅ Step 2: 构造 skip-gram 样本
skip_grams = [skipgrams(wid, vocabulary_size=vocab_size, window_size=10) for wid in wids]

# ✅ Step 3: 建立模型
from keras.models import Model, Sequential
from keras.layers import Embedding, Reshape, Dot, Dense

word_model = Sequential()
word_model.add(Embedding(vocab_size, embed_size, input_length=1))
word_model.add(Reshape((embed_size,)))

context_model = Sequential()
context_model.add(Embedding(vocab_size, embed_size, input_length=1))
context_model.add(Reshape((embed_size,)))

merged = Dot(axes=1)([word_model.output, context_model.output])
output = Dense(1, activation="sigmoid")(merged)
model = Model([word_model.input, context_model.input], output)
model.compile(loss="mean_squared_error", optimizer="rmsprop")

# ✅ Step 4: 训练模型
for epoch in range(1, 6):
    loss = 0
    for i, elem in enumerate(skip_grams):
        if elem == ([], []): continue
        pair_first = np.array(list(zip(*elem[0]))[0])
        pair_second = np.array(list(zip(*elem[0]))[1])
        labels = np.array(elem[1])
        loss += model.train_on_batch([pair_first, pair_second], labels)
    print('Epoch:', epoch, 'Loss:', loss)

# ✅ Step 5: 提取词向量
word_embed_layer = model.layers[2]
weights = word_embed_layer.get_weights()[0][1:]  # 跳过 index 0
